# Lab 3.1 &mdash; Memory That Survives a Long Session

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Watch an agent forget, by growing the history until the window bites
- Bound it with <code>trim_messages</code> &mdash; including the counter that fails here
- Summarise what you drop, so compaction is not amnesia
- Hand the whole problem to a checkpointer and a <code>thread_id</code>

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 3 labs work one case: payment exceptions on a small
> synthetic ledger. Modules 1 and 2 built the agent; Module 3 gives it a memory and a
> state you can inspect.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 3 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

Everything an agent "remembers" is something your code put back in front of it. That leaves three
jobs, and LangChain has a piece for each:

| Job | The piece |
|---|---|
| keep the window bounded | `trim_messages` |
| keep what you dropped | a summary chain |
| keep it across turns and restarts | a checkpointer + `thread_id` |

Do the first without the second and you have built amnesia with a token budget.

## Section 1 &mdash; Find the turn where it forgets

A long investigation, one important fact stated at the very beginning. Grow the history until
the fact falls out of the window, and note which turn it happened on.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.messages.utils import count_tokens_approximately

SYSTEM = ("You are a payments operations analyst working one case. Answer only from this "
          "conversation.")

THE_FACT = "The client contact for this case is Priya Raman on the Singapore desk."

def long_session(turns: int = 14) -> list:
    """A conversation whose first human turn carries the fact that matters."""
    msgs = [SystemMessage(SYSTEM), HumanMessage(f"Open the case for PMT-1005. {THE_FACT}")]
    for i in range(turns):
        msgs.append(AIMessage(f"Noted. Checking sanction screening batch {i}: "
                              + "reviewing the counterparty records in detail. " * 6))
        msgs.append(HumanMessage(f"And what about batch {i + 1}?"))
    return msgs


def window_fits(messages: list, budget: int) -> bool:
    """Does this conversation still fit the budget?"""
    return count_tokens_approximately(messages) <= budget

In [ ]:
# --- Self-check: Section 1   (counting only -- no model call)
BUDGET = 400

check("a short session fits",
      lambda: window_fits(long_session(0), BUDGET) is True)
check("a long session does not",
      lambda: window_fits(long_session(14), BUDGET) is False)
check("the count grows with the conversation",
      lambda: count_tokens_approximately(long_session(10))
              > count_tokens_approximately(long_session(2)))
check("the fact is in the session to begin with",
      lambda: any("Priya" in str(m.content) for m in long_session(14)),
      "everything below is about whether it is still there LATER")

def _first_overflow():
    for n in range(0, 20):
        if not window_fits(long_session(n), BUDGET):
            return n
    return None
guard(lambda: print(f"\nthe window overflows at turn {_first_overflow()} "
                    f"on a {BUDGET}-token budget"))

## Section 2 &mdash; Bound it, and keep what you drop

`trim_messages` bounds the window. On its own that is amnesia: the fact from turn one is simply
gone. So summarise what you are about to drop and put the summary back as a system message.

**The counter matters.** The obvious `token_counter=llm` raises `NotImplementedError` here &mdash;
`langchain-openai` can only count for models `tiktoken` has an encoding for, and a
gateway-served model is not one. Use `count_tokens_approximately`, a plain function over the text.

In [ ]:
from langchain_core.messages import trim_messages

def bounded(messages: list, budget: int = 400) -> list:
    """Keep the system message and as many recent turns as fit."""
    return trim_messages(
        messages, max_tokens=budget,
        token_counter=BLANK,          # TODO: which counter works for a gateway-served model?
        strategy="last", include_system=True, start_on="human", allow_partial=False)


def compact(messages: list, budget: int = 400) -> list:
    """Trim, then put a summary of what was dropped back in front of what survived."""
    kept = bounded(messages, budget)
    kept_ids = {id(m) for m in kept}
    dropped = [m for m in messages if id(m) not in kept_ids and m.type != "system"]
    if not dropped:
        return kept
    summary = summarise(dropped)
    head = [m for m in kept if m.type == "system"]
    tail = [m for m in kept if m.type != "system"]
    return head + [SystemMessage("Earlier in this case: " + summary)] + tail


def summarise(dropped: list) -> str:
    """One line covering the turns that are about to be discarded."""
    if not llm_ready():
        return " ".join(str(m.content) for m in dropped)[:300]
    joined = "\n".join(f"{m.type}: {m.content}" for m in dropped)[:4000]
    return ask("Summarise these earlier turns in two sentences. Preserve every proper noun, "
               "reference number and named person exactly.\n\n" + joined)

In [ ]:
# --- Self-check: Section 2   (trim_messages is pure; summarise degrades offline)
_long = long_session(14)

check("trimming bounds the window",
      lambda: window_fits(bounded(_long), 420) is True,
      "trim_messages needs a token_counter it can actually call on this model")
check("trimming really dropped turns",
      lambda: len(bounded(_long)) < len(_long))
check("the system message survives",
      lambda: bounded(_long)[0].type == "system",
      "include_system=True -- dropping the instructions is the worst possible trim")
check("what survives is the END of the conversation",
      lambda: bounded(_long)[-1].content == _long[-1].content,
      'strategy="last" keeps recent turns; "first" would keep the stale ones')
check("plain trimming LOSES the fact from turn one",
      lambda: not any("Priya" in str(m.content) for m in bounded(_long)),
      "this is the point of the section -- a bounded window is amnesia unless you do more")
check("compaction puts a summary back",
      lambda: sum(1 for m in compact(_long) if m.type == "system") == 2,
      "one system message for the instructions, one for what was dropped")
check("a short session is left alone",
      lambda: len(compact(long_session(0))) == len(long_session(0)))

## Section 3 &mdash; Or: let a checkpointer do it

Everything above is what you write when you are managing the message list yourself. Attach a
**checkpointer** to `create_agent` and the history is stored for you, keyed by `thread_id` &mdash;
across turns, across restarts, and separately per case.

The two approaches are not rivals. The checkpointer decides *where the history lives*; trimming
decides *how much of it you send*. Real systems do both.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1005'."""
    rec = LEDGER.get(ref)
    return json.dumps({"ref": ref, **rec}) if rec else f"no payment found with reference {ref!r}"

def remembering_agent():
    """An agent whose history is kept for it, per thread."""
    return create_agent(model=get_llm(), tools=[lookup_payment], system_prompt=SYSTEM,
                        checkpointer=InMemorySaver())

def thread(case_ref: str) -> dict:
    """The config that selects which conversation this call belongs to."""
    return {"configurable": {"thread_id": BLANK}}   # TODO: what separates one case from another?

In [ ]:
# --- Self-check: Section 3   (config shape only -- no model call)
check("the thread config has the shape LangGraph expects",
      lambda: set(thread("PMT-1005")) == {"configurable"})
check("the thread is keyed by the case",
      lambda: thread("PMT-1005")["configurable"]["thread_id"] == "PMT-1005")
check("two cases get two threads",
      lambda: thread("PMT-1005") != thread("PMT-1003"),
      "one thread_id for everything is how one client sees another client's case")

## Run it for real &mdash; forgetting, and not forgetting

In [ ]:
if llm_ready():
    def _forget():
        session = long_session(14)
        question = "Who is the client contact for this case?"

        naive = bounded(session) + [HumanMessage(question)]
        kept  = compact(session)  + [HumanMessage(question)]

        print("--- trimmed only ---")
        print(get_llm().invoke(naive).content[:220])
        print("\n--- trimmed, with a summary of what was dropped ---")
        print(get_llm().invoke(kept).content[:220])
        print("\nsummary that was carried forward:")
        print("  " + next(m.content for m in kept if "Earlier in this case" in str(m.content))[:300])
    guard(_forget)

## Run it for real &mdash; the checkpointer

In [ ]:
if llm_ready():
    def _threads():
        agent = remembering_agent()
        agent.invoke({"messages": [HumanMessage("Open PMT-1005. The contact is Priya Raman.")]},
                     thread("PMT-1005"))
        agent.invoke({"messages": [HumanMessage("Open PMT-1003. The contact is Lee Wei.")]},
                     thread("PMT-1003"))

        for ref in ("PMT-1005", "PMT-1003"):
            out = agent.invoke({"messages": [HumanMessage("Who is the contact for this case?")]},
                               thread(ref))
            print(f"{ref}: {out['messages'][-1].content[:120]}")
            print(f"          {len(out['messages'])} messages on this thread")
    guard(_threads)

### Read it

**The first pair.** Trimming alone answers the contact question wrongly or not at all &mdash; Priya
Raman fell out of the window twelve turns ago. Trimming *with a summary* still has her, in one
line instead of twenty. That is the difference between compaction and amnesia, and it is why
`summarise` is told to preserve proper nouns exactly: a summary that paraphrases names has thrown
away the only part that mattered.

**The second pair.** Two threads, two histories, no code of yours managing either. Each `thread_id`
is a separate conversation, and the agent answered each from its own. Note what would happen if
`thread()` returned a constant: every case would append to one history, and the second client's
contact would be answered with the first client's name. That is not a hypothetical bug &mdash; it is
the single commonest way agent memory leaks between users.

Neither approach removes the need for the other. The checkpointer stores everything forever;
`trim_messages` decides how much of it you pay to send on this turn.

In [ ]:
score()

## Your turn

1. Put the two together: trim the messages the checkpointed agent sends without deleting them
   from the thread. (`create_agent` takes middleware for this; failing that, trim before you
   pass them in.) Confirm the thread still has the full history afterwards.
2. Change `summarise` to drop the "preserve every proper noun" instruction and re-run. Count how
   many turns it takes before the contact's name is gone. That sentence is the whole safeguard.
3. Swap `InMemorySaver` for `SqliteSaver` writing to a file under `WORK`, restart the kernel, and
   ask the follow-up question again. Lab 3.4 is about what that buys you.